Code based on Svet's notebook

In [1]:
import glob
import gc
import cudf
from numba import cuda
import pandas as pd
from tqdm import tqdm
import os

os.chdir('/home/rapids/Capstone')


In [7]:
# Function to convert a CSV file to a Parquet file
def convert_csv_to_parquet(file_path):
    # Read the CSV file
    df = cudf.read_csv(file_path)

    # Construct the new file path for the Parquet file
    parquet_file_path = file_path.replace('.csv', '.parquet')

    # Write the DataFrame to a Parquet file
    df.to_parquet(parquet_file_path, index=False)

    # Delete the DataFrame to free up memory
    del df

# Get the list of CSV files
file_list = glob.glob('data/*.csv')
print(f"Found files: {file_list}")

# Process files with a progress bar
for file_path in tqdm(file_list, desc="Converting CSV to Parquet"):
    convert_csv_to_parquet(file_path)

    
    

Found files: ['data/2023_01_Gener_BicingNou_ESTACIONS.csv', 'data/2021_10_Octubre_BicingNou_ESTACIONS.csv', 'data/2024_04_Abril_BicingNou_ESTACIONS.csv', 'data/2024_03_Marc_BicingNou_ESTACIONS.csv', 'data/2022_01_Gener_BicingNou_ESTACIONS.csv', 'data/2023_11_Novembre_BicingNou_ESTACIONS.csv', 'data/2022_12_Desembre_BicingNou_ESTACIONS.csv', 'data/2024_02_Febrer_BicingNou_ESTACIONS.csv', 'data/2022_05_Maig_BicingNou_ESTACIONS.csv', 'data/2022_10_Octubre_BicingNou_ESTACIONS.csv', 'data/2024_05_Maig_BicingNou_ESTACIONS.csv', 'data/2020_01_Gener_BicingNou_ESTACIONS.csv', 'data/2022_09_Setembre_BicingNou_ESTACIONS.csv', 'data/2023_07_Juliol_BicingNou_ESTACIONS.csv', 'data/2023_08_Agost_BicingNou_ESTACIONS.csv', 'data/2021_11_Novembre_BicingNou_ESTACIONS.csv', 'data/2021_12_Desembre_BicingNou_ESTACIONS.csv', 'data/2022_04_Abril_BicingNou_ESTACIONS.csv', 'data/2020_08_Agost_BicingNou_ESTACIONS.csv', 'data/2023_02_Febrer_BicingNou_ESTACIONS.csv', 'data/2020_05_Maig_BicingNou_ESTACIONS.csv', 'd

Converting CSV to Parquet: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:26<00:00,  1.99it/s]


In [8]:
# For each parquet file, read it and, if present, delete the columns 'traffic', 'V1' and 'last_updated'
def clean_parquet_files():
    # For each parquet file, read it and, if present, delete the columns 'traffic', 'V1' and 'last_updated'
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Cleaning Parquet Files"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Drop the columns if they exist
        columns_to_drop = ['traffic', 'V1', 'last_updated']
        for column in columns_to_drop:
            if column in df.columns:
                df.drop(column, axis=1, inplace=True)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
clean_parquet_files()

Cleaning Parquet Files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:13<00:00,  3.81it/s]


In [9]:
# For every parquet file, replace any negative value in the numeric columns with zero


def replace_negative_values_with_zero():
    # For every parquet file, replace any negative value in the numeric columns with zero
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Replacing Negative Values"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Replace negative values with zero
        for column in df.columns:
            if df[column].dtype in ['int8', 'int16', 'int32', 'int64', 'float32', 'float64']:
                df[column] = df[column].clip(lower=0)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
replace_negative_values_with_zero()

Replacing Negative Values: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:14<00:00,  3.56it/s]


In [10]:
# Convert the 'last_reported' column from Unix timestamp to datetime. After that, sort the dataframe in ascending order. For each year, day and hour, create a new column with the corresponding value. Finally, reset the index of the DataFrame and write it back to the Parquet file.

def convert_last_reported_to_datetime():
    # Convert the 'last_reported' column from Unix timestamp to datetime
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Converting 'last_reported' to Datetime"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Convert the 'last_reported' column from Unix timestamp to datetime
        df['last_reported'] = cudf.to_datetime(df['last_reported'], unit='s')

        # Sort the DataFrame in ascending order
        df = df.sort_values('last_reported')

        # Create new columns for year, day and hour
        df['year'] = df['last_reported'].dt.year
        df['day'] = df['last_reported'].dt.day
        df['hour'] = df['last_reported'].dt.hour

        # Reset the index of the DataFrame
        df.reset_index(drop=True, inplace=True)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
convert_last_reported_to_datetime()

Converting 'last_reported' to Datetime:   0%|                                                                                                  | 0/53 [00:00<?, ?it/s]

Converting 'last_reported' to Datetime: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:18<00:00,  2.94it/s]


In [11]:
# Delete the last reported column

def delete_last_reported_column():
    # Delete the last reported column
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Deleting 'last_reported' Column"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Drop the last reported column
        df.drop('last_reported', axis=1, inplace=True)

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
delete_last_reported_column()

Deleting 'last_reported' Column: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:09<00:00,  5.43it/s]


In [ ]:
# For each parque file, for each year, day and hour, calculate the mode of columns 'station_id', 'is_installed', 'is_renting', 'is_returning' and 'is_charging_station'. For the rest of the columns calculate the average. Write the resulting DataFrame back to the Parquet file.

def calculate_aggregates():
    # For each parquet file, import it to a dataframe
    for file_path in tqdm(glob.glob('data/*.parquet'), desc="Average of values per hour"):
        # Read the Parquet file
        df = cudf.read_parquet(file_path)

        # Create a dataframe without status and is_charging_station
        dfa = df.drop(['status', 'is_charging_station'], axis=1)
        # Group by year, day and hour and station_id and calculate average
        dfa = dfa.groupby(['year', 'day', 'hour', 'station_id']).mean().reset_index()

        # Create a dataframe with only status and is_charging_station
        dfb = df[['year', 'day', 'hour', 'station_id', 'status', 'is_charging_station']]
        # Encode the status and is_charging_station columns and make them numerical
        print(f"Encoding 'status' column: {dfb['status'].unique()} to numerical categories")
        dfb['status'] = dfb['status'].astype('category').cat.codes
        print(f"Encoding 'is_charging_station' column: {dfb['is_charging_station'].unique()} to numerical categories")
        dfb['is_charging_station'] = dfb['is_charging_station'].astype('category').cat.codes
        
        # Group by year, day and hour and station_id and calculate mean
        dfb = dfb.groupby(['year', 'day', 'hour', 'station_id']).mean().reset_index()
        # In the status and is_charging_station columns, round the values to the nearest integer
        dfb['status'] = dfb['status'].round().astype('int')
        dfb['is_charging_station'] = dfb['is_charging_station'].round().astype('int')

        # Merge the two dataframes
        df = cudf.merge(dfa, dfb, on=['year', 'day', 'hour', 'station_id'])

        # Write the DataFrame back to the Parquet file
        df.to_parquet(file_path, index=False)

        # Delete the DataFrame to free up memory
        del df

# Call the function
calculate_aggregates()

Average of values per hour:   0%|                                                                                                              | 0/53 [00:00<?, ?it/s]

Encoding 'status' column: 0        IN_SERVICE
1    NOT_IN_SERVICE
2       MAINTENANCE
3           PLANNED
Name: status, dtype: object to numerical categories
Encoding 'is_charging_station' column: 0    True
Name: is_charging_station, dtype: bool to numerical categories


Average of values per hour:   2%|█▉                                                                                                    | 1/53 [00:00<00:39,  1.33it/s]

        year  day  hour  station_id  num_bikes_available  \
0       2022   24     7         164            10.583333   
1       2022    2    21         493             5.166667   
2       2022   24    16         258             4.833333   
3       2022   21    16         371             7.416667   
4       2022   18     7           1            12.833333   
...      ...  ...   ...         ...                  ...   
341884  2022   12     0         124             9.666667   
341885  2022    9    22         260            12.714286   
341886  2022    7    23         122            20.500000   
341887  2022    2    11          95             9.583333   
341888  2022   26     1         442            15.666667   

        num_bikes_available_types.mechanical  num_bikes_available_types.ebike  \
0                                   2.166667                         8.416667   
1                                   0.000000                         5.166667   
2                                   

Average of values per hour:   4%|███▊                                                                                                  | 2/53 [00:01<00:25,  2.01it/s]

...

Encoding 'status' column: 0        IN_SERVICE
1    NOT_IN_SERVICE
2       MAINTENANCE
Name: status, dtype: object to numerical categories
Encoding 'is_charging_station' column: 0    True
Name: is_charging_station, dtype: bool to numerical categories
        year  day  hour  station_id  num_bikes_available  \
0       2022   18     6         371            32.000000   
1       2022   21    16         519             0.666667   
2       2022    9     0          10            22.000000   
3       2022    1    20         161             3.333333   
4       2022   14    16         398             8.666667   
...      ...  ...   ...         ...                  ...   
375845  2022   24     0         393            10.666667   
375846  2022    1    16         188            19.833333   
375847  2022    7     5         308             6.000000   
375848  2022   29    19         121             1.500000   
375849  2022    5    16         478             8.583333   

        num_bikes_available_t

Average of values per hour:  98%|███████████████████████████████████████████████████████████████████████████████████████████████████  | 52/53 [00:14<00:00,  3.69it/s]

Encoding 'status' column: 0        IN_SERVICE
1    NOT_IN_SERVICE
2       MAINTENANCE
3           PLANNED
Name: status, dtype: object to numerical categories
Encoding 'is_charging_station' column: 0     True
1    False
Name: is_charging_station, dtype: bool to numerical categories
        year  day  hour  station_id  num_bikes_available  \
0       2020    4     6         488             0.000000   
1       2020   12    18         197             1.000000   
2       2020   14    17         438             8.272727   
3       2020   10    14         162            16.416667   
4       2020   21    20         324             1.083333   
...      ...  ...   ...         ...                  ...   
323118  2020    6    15         133            10.333333   
323119  2020    2     1         218            28.583333   
323120  2020    3    15          88             0.583333   
323121  2020   14    10         379            12.833333   
323122  2020    1    19         489             0.000000  

Average of values per hour: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:14<00:00,  3.56it/s]

Encoding 'status' column: 0           PLANNED
1        IN_SERVICE
2    NOT_IN_SERVICE
3       MAINTENANCE
Name: status, dtype: object to numerical categories
Encoding 'is_charging_station' column: 0    True
Name: is_charging_station, dtype: bool to numerical categories
        year  day  hour  station_id  num_bikes_available  \
0       2021   24     4         428             1.000000   
1       2021    6    22         327             1.000000   
2       2021    4     0          40            14.000000   
3       2021    2     0         226             2.000000   
4       2021   27     7         176             6.833333   
...      ...  ...   ...         ...                  ...   
375723  2021   25     6         184            13.250000   
375724  2021    1     5          62             5.000000   
375725  2021    6    13         395            15.666667   
375726  2021    4     1         351            10.000000   
375727  2021   19     6         414            26.583333   

        n

In [8]:
# Define excel file
excel_file = 'Informacio_Estacions_Bicing_2025.xlsx'

# Load excel file
df = pd.read_excel(excel_file)

# Ensure 'station_id' and 'capacity' are integers
df['station_id'] = df['station_id'].astype('int')
df['capacity'] = df['capacity'].astype('int')

df


,station_id,name,physical_configuration,lat,lon,altitude,address,cross_street,post_code,capacity,is_charging_station,short_name,nearby_distance,_ride_code_support,rental_uris,is_valet_station
0,1,"GRAN VIA CORTS CATALANES, 760",ELECTRICBIKESTATION,41.397978,2.180107,16.0,"GRAN VIA CORTS CATALANES, 760",02-Eixample/05-el Fort Pienc,8013.0,46,1.0,1.0,1000.0,1.0,NaN,NaN
1,2,"C/ ROGER DE FLOR, 126",ELECTRICBIKESTATION,41.395488,2.177198,17.0,"C/ ROGER DE FLOR, 126",02-Eixample/05-el Fort Pienc,8013.0,29,1.0,2.0,1000.0,1.0,NaN,NaN
2,3,"C/ NÀPOLS, 82",ELECTRICBIKESTATION,41.394156,2.181331,11.0,"C/ NÀPOLS, 82",02-Eixample/05-el Fort Pienc,8013.0,27,1.0,3.0,1000.0,1.0,NaN,NaN
3,4,"C/ RIBES, 13",ELECTRICBIKESTATION,41.393317,2.181248,8.0,"C/ RIBES, 13",02-Eixample/05-el Fort Pienc,8013.0,21,1.0,4.0,1000.0,1.0,NaN,NaN
4,5,"PG. LLUIS COMPANYS, 11 (ARC TRIOMF)",ELECTRICBIKESTATION,41.391103,2.180176,7.0,"PG. LLUIS COMPANYS, 11 (ARC TRIOMF)","01-CiutatVella/04-Sant Pere, Santa Caterina i ...",8018.0,39,1.0,5.0,1000.0,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,540,"C/ MANDONI, 6",ELECTRICBIKESTATION,41.369348,2.142601,18.0,"C/ MANDONI, 6",03-Sants-Montjuïc/14-la Font de la Guatlla,8004.0,20,1.0,521.0,1.0,1.0,NaN,NaN
529,541,"C/ MARBRE, 2",ELECTRICBIKESTATION,41.361416,2.147617,28.0,"C/ MARBRE, 2",03-Sants-Montjuïc/13-la Marina de Port,8038.0,28,1.0,524.0,1000.0,1.0,NaN,NaN
530,542,Copa América Barcelona - 542,VALET,41.374538,2.189217,NaN,Copa América Barcelona 2024,NaN,NaN,1,0.0,9000.0,1.0,1.0,NaN,0.0
531,543,Copa América Barcelona - 543,VALET,41.383830,2.191371,NaN,Copa América Barcelona 2024,NaN,NaN,1,0.0,9001.0,1.0,1.0,NaN,0.0


In [7]:
!pip install openpyxl

In [4]:
df = cudf.read_parquet('data/2022_11_Novembre_BicingNou_ESTACIONS.parquet')
df.dtypes

year                                      int16
day                                       int16
hour                                      int16
station_id                                int64
num_bikes_available                     float64
num_bikes_available_types.mechanical    float64
num_bikes_available_types.ebike         float64
num_docks_available                     float64
is_installed                            float64
is_renting                              float64
is_returning                            float64
ttl                                     float64
status                                    int64
is_charging_station                       int64
dtype: object

In [5]:
del df
# del grouped
# del average_df
gc.collect()

cuda.current_context().deallocations.clear()
# cuda.select_device(0)
# cuda.close()

In [ ]:
# Group by year, day and hour
# grouped = df.groupby(['year', 'day', 'hour'])

In [ ]:
# grouped.head(20)

MemoryError: std::bad_alloc: out_of_memory: CUDA error at: /opt/conda/include/rmm/mr/device/cuda_memory_resource.hpp

In [ ]:
# !git config --global user.email "github@botello.me"
# !git config --global user.name "LEBsci"

/usr/bin/sh: 1: git: not found
/usr/bin/sh: 1: git: not found
